In [23]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [24]:

file_path = "/Users/sabari/Documents/DS/GUVI/dspatrol/PatrolIQ-Smart_Safety_Analytics_Platform-main/data/Crime_Clean.csv"

df = pd.read_csv(file_path)

print(df.shape)
df.head()

(494426, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,13307269,JG540388,12/14/2023 12:01:00 AM,001XX S TROY ST,0820,THEFT,$500 AND UNDER,VACANT LOT / LAND,False,False,...,28.0,27.0,06,1155371.0,1898949.0,2023,12/22/2023 03:41:15 PM,41.878520,-87.704974,"(41.878519796, -87.704973708)"
1,11339480,JB295964,06/06/2018 05:03:00 PM,003XX S HOMAN AVE,1330,CRIMINAL TRESPASS,TO LAND,SIDEWALK,True,False,...,28.0,27.0,26,1153781.0,1897987.0,2018,06/13/2018 04:15:47 PM,41.875912,-87.710838,"(41.875911762, -87.710837503)"
2,2856772,HJ520501,07/25/2003 04:45:00 PM,001XX E DELAWARE PL,0870,THEFT,POCKET-PICKING,HOTEL/MOTEL,False,False,...,42.0,8.0,06,1177786.0,1906673.0,2003,02/28/2018 03:56:25 PM,41.899235,-87.622436,"(41.899235024, -87.622436193)"
3,11912488,JC535868,11/14/2019 08:00:00 AM,031XX S SHIELDS AVE,1110,DECEPTIVE PRACTICE,BOGUS CHECK,"SCHOOL, PRIVATE, BUILDING",False,False,...,11.0,34.0,11,1174469.0,1884050.0,2019,12/08/2019 03:57:40 PM,41.837231,-87.635295,"(41.837230767, -87.635295009)"
4,10941360,JA259248,05/11/2017 11:18:00 AM,008XX N LOCKWOOD AVE,1821,NARCOTICS,MANU/DEL:CANNABIS 10GM OR LESS,SIDEWALK,True,False,...,37.0,25.0,18,1140872.0,1904987.0,2017,02/10/2018 03:50:01 PM,41.895368,-87.758063,"(41.895367769, -87.75806298)"


In [25]:
df['Date'] = pd.to_datetime(df['Date'])

/var/folders/pp/g4gxxt1n4msf_38rzlmv0c9w0000gn/T/ipykernel_32524/2394721818.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])


In [26]:
df = df[df['Latitude'] >= 37]

In [27]:
df['Hour'] = df['Date'].dt.hour
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.month

In [28]:
df['Is_Weekend'] = df['Day_of_Week'].isin(['Saturday','Sunday']).astype(int)

In [29]:
def get_season(month):
    if month in [12,1,2]:
        return "Winter"
    elif month in [3,4,5]:
        return "Spring"
    elif month in [6,7,8]:
        return "Summer"
    else:
        return "Fall"

df['Season'] = df['Month'].apply(get_season)

In [30]:
high_severity = ['HOMICIDE','KIDNAPPING','CRIM SEXUAL ASSAULT','ROBBERY','ARSON']
medium_severity = ['BATTERY','ASSAULT','BURGLARY','MOTOR VEHICLE THEFT']

def severity_score(crime):
    if crime in high_severity:
        return 3
    elif crime in medium_severity:
        return 2
    else:
        return 1

df['Crime_Severity_Score'] = df['Primary Type'].apply(severity_score)

In [31]:
le = LabelEncoder()

df['Primary_Type_Encoded'] = le.fit_transform(df['Primary Type'])
df['Location_Encoded'] = le.fit_transform(df['Location Description'])
df['Season_Encoded'] = le.fit_transform(df['Season'])
df['Day_Encoded'] = le.fit_transform(df['Day_of_Week'])

In [32]:
features = [
    'Latitude',
    'Longitude',
    'Hour',
    'Month',
    'Is_Weekend',
    'Crime_Severity_Score',
    'Primary_Type_Encoded',
    'Location_Encoded',
    'Season_Encoded',
    'Day_Encoded',
    'Arrest',
    'Domestic'
]

df_model = df[features]

df_model.head()

,Latitude,Longitude,Hour,Month,Is_Weekend,Crime_Severity_Score,Primary_Type_Encoded,Location_Encoded,Season_Encoded,Day_Encoded,Arrest,Domestic
0,41.878520,-87.704974,0,12,0,1,31,158,3,4,False,False
1,41.875912,-87.710838,17,6,0,1,8,145,2,6,True,False
2,41.899235,-87.622436,16,7,0,1,31,94,2,0,False,False
3,41.837231,-87.635295,8,11,0,1,9,141,0,4,False,False
4,41.895368,-87.758063,11,5,0,1,18,145,1,4,True,False


In [33]:
scaler = StandardScaler()

scaled_data = scaler.fit_transform(df_model)

df_scaled = pd.DataFrame(scaled_data, columns=features)

df_scaled.head()

,Latitude,Longitude,Hour,Month,Is_Weekend,Crime_Severity_Score,Primary_Type_Encoded,Location_Encoded,Season_Encoded,Day_Encoded,Arrest,Domestic
0,0.414795,-0.574671,-1.934739,1.623297,-0.623667,-0.768661,1.388772,0.959545,1.395977,0.504356,-0.580103,-0.457865
1,0.384649,-0.674220,0.580297,-0.166550,-0.623667,-0.768661,-0.572140,0.692181,0.483884,1.496725,1.723832,-0.457865
2,0.654239,0.826550,0.432354,0.131758,-0.623667,-0.768661,1.388772,-0.356712,0.483884,-1.480381,-0.580103,-0.457865
3,-0.062458,0.608249,-0.751193,1.324989,-0.623667,-0.768661,-0.486883,0.609915,-1.340302,0.504356,-0.580103,-0.457865
4,0.609538,-1.475956,-0.307363,-0.464858,-0.623667,-0.768661,0.280431,0.692181,-0.428209,0.504356,1.723832,-0.457865


In [34]:
df_scaled.to_csv("/Users/sabari/Documents/DS/GUVI/dspatrol/PatrolIQ-Smart_Safety_Analytics_Platform-main/data/crime_featured_scaled.csv", index=False)

print("Feature Engineered Dataset Saved ✅")

Feature Engineered Dataset Saved ✅
